# Saúde+ Analytics — Case Técnico Completo
**Análise de Dados de Prescrições Digitais**

Este notebook consolida todas as análises para o case Saúde+, respondendo às 7 perguntas strategicamente propostas com insights acionáveis e recomendações operacionais.

**Estrutura:**
1. Setup e Carregamento de Dados
2. Exploração e Limpeza
3. Q1 — Prescrições Diárias & Sazonalidade
4. Q2 — Pacientes Atendidos
5. Q3 — Especialidades Médicas
6. Q4 — Open Rate
7. Q5 — Conversão por Canal
8. Q6 — Análise de Medicamentos
9. Q7 — Insights Operacionais & Recomendações

## 1. Setup & Importações

In [19]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os
from datetime import datetime

# Configurar DuckDB
con = duckdb.connect()

# Diretório de dados
DATA_DIR = './data'

print('✓ Bibliotecas importadas com sucesso')
print(f'✓ DuckDB conectado | Diretório: {DATA_DIR}')

✓ Bibliotecas importadas com sucesso
✓ DuckDB conectado | Diretório: ./data


## 2. Carregamento de Dados como Views DuckDB

In [21]:
# Carregar CSVs como views DuckDB
con.execute(f"""
CREATE OR REPLACE VIEW prescricao AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/prescricaomedicamento.csv', header=true);
""")

con.execute(f"""
CREATE OR REPLACE VIEW medicamentos AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/medicamentos.csv', header=true);
""")

con.execute(f"""
CREATE OR REPLACE VIEW medicos AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/medicos.csv', header=true);
""")

# Verificar carregamento
presc_count = con.execute('SELECT COUNT(*) FROM prescricao').fetchone()[0]
med_count = con.execute('SELECT COUNT(*) FROM medicamentos').fetchone()[0]
doc_count = con.execute('SELECT COUNT(*) FROM medicos').fetchone()[0]

print(f'✓ Dados carregados:')
print(f'  - Prescrições: {presc_count:,} linhas')
print(f'  - Medicamentos: {med_count:,} linhas')
print(f'  - Médicos: {doc_count:,} linhas')

✓ Dados carregados:
  - Prescrições: 73,929 linhas
  - Medicamentos: 22 linhas
  - Médicos: 45,025 linhas


## 3. Exploração e Validação dos Dados

In [3]:
# Amostra da estrutura dos dados
# print("=" * 80)
# print("ESTRUTURA DA TABELA PRESCRIÇÃO")
# print("=" * 80)
# df_presc_sample = con.execute("""
#     SELECT column_name, column_type
#     FROM information_schema.columns
#     WHERE table_name = 'prescricao'
#     ORDER BY ordinal_position
# """).df()
# print(df_presc_sample.to_string(index=False))

print("\n" + "=" * 80)
print("AMOSTRA DE DADOS (5 primeiras linhas)")
print("=" * 80)
df_sample = con.execute("""
    SELECT 
        idprescricao, idpaciente, idmedico, idmedicamento,
        dataprescricao, visualizadapaciente, itemvendido, canalvenda
    FROM prescricao
    LIMIT 5
""").df()
print(df_sample.to_string(index=False))

# Período disponível
date_range = con.execute("""
    SELECT 
        MIN(CAST(dataprescricao AS DATE)) AS data_inicio,
        MAX(CAST(dataprescricao AS DATE)) AS data_fim,
        COUNT(DISTINCT CAST(dataprescricao AS DATE)) AS dias_com_movimento
    FROM prescricao
""").df()

print("\n" + "=" * 80)
print("PERÍODO DE DADOS DISPONÍVEL")
print("=" * 80)
print(date_range.to_string(index=False))


AMOSTRA DE DADOS (5 primeiras linhas)
 idprescricao  idpaciente  idmedico  idmedicamento      dataprescricao  visualizadapaciente  itemvendido      canalvenda
     65286505       16277    115888           3783 2025-01-28 13:00:11                 True            0  não convertido
     65053805       34330     41433          14714 2025-01-24 13:02:29                False            0  não convertido
     69631101       10831     77184           3783 2025-03-26 15:00:26                 True            1 farmacia fisica
     67746000       54414    107252          11191 2025-02-28 23:31:13                False            0  não convertido
     68167102       63594     30764           3783 2025-03-08 13:35:08                 True            0  não convertido

PERÍODO DE DADOS DISPONÍVEL
data_inicio   data_fim  dias_com_movimento
 2025-01-01 2025-04-18                 108


---
## Q1: Quantas prescrições foram emitidas diariamente? Existe sazonalidade?

**O que analisamos:**
- Volume diário de prescrições (tendência geral)
- Padrões semanais e sazonalidade
- Dias com picos de atividade

**Por que importa:**
Entender sazonalidade ajuda a otimizar recursos (atendimento, suporte, farmácias parceiras).

In [13]:
# Q1.1: Volume diário de prescrições
df_daily = con.execute("""
SELECT
    CAST(dataprescricao AS DATE) AS data,
    COUNT(DISTINCT idprescricao) AS prescricoes
FROM prescricao
GROUP BY 1
ORDER BY 1
""").df()

# Q1.2: Estatísticas gerais (corrigido com CTE para evitar aggregate aninhado)
stats_summary = {
    'Total': con.execute("SELECT COUNT(DISTINCT idprescricao) FROM prescricao").fetchone()[0],
    'Dias': con.execute("SELECT COUNT(DISTINCT CAST(dataprescricao AS DATE)) FROM prescricao").fetchone()[0],
    'Média Diária': round(con.execute("SELECT COUNT(DISTINCT idprescricao) * 1.0 / COUNT(DISTINCT CAST(dataprescricao AS DATE)) FROM prescricao").fetchone()[0], 1),
    'Min': df_daily['prescricoes'].min(),
    'Max': df_daily['prescricoes'].max()
}

print("📊 ESTATÍSTICAS DE PRESCRIÇÕES DIÁRIAS")
print("=" * 60)
for k, v in stats_summary.items():
    print(f"{k:20s}: {v:>10,}" if isinstance(v, int) else f"{k:20s}: {v:>10}")

# Q1.3: Sazonalidade por dia da semana
df_dow = con.execute("""
SELECT
    DAYNAME(dataprescricao::TIMESTAMP) AS dia_semana,
    CASE DAYOFWEEK(dataprescricao::TIMESTAMP)
        WHEN 1 THEN 0 WHEN 2 THEN 1 WHEN 3 THEN 2 WHEN 4 THEN 3 
        WHEN 5 THEN 4 WHEN 6 THEN 5 WHEN 7 THEN 6
    END AS num_dia,
    COUNT(DISTINCT idprescricao) AS prescricoes
FROM prescricao
GROUP BY 1, 2
ORDER BY 2
""").df()

print("\n📅 SAZONALIDADE POR DIA DA SEMANA")
print("=" * 60)
print(df_dow.to_string(index=False))

# Gráfico: Tendência diária
print("\n📈 GRÁFICO: Tendência Diária de Prescrições")
print("   → Tipo: Linha temporal")
print("   → Eixo X: Data da prescrição")
print("   → Eixo Y: Número de prescrições por dia")
print("   → Insight: Mostra a variação diária do volume, útil para identificar sazonalidade e picos de demanda")
try:
    fig_daily = px.line(df_daily, x='data', y='prescricoes',
                        title='Tendência de Prescrições Diárias',
                        labels={'data': 'Data', 'prescricoes': 'Prescrições'},
                        markers=True, template='plotly_white')
    fig_daily.update_layout(height=400)
    fig_daily.write_html('/tmp/prescrições_diárias.html')
    print("✓ Gráfico salvo: /tmp/prescrições_diárias.html")
    fig_daily.show()
except Exception as e:
    print(f"\n⚠️ Erro ao renderizar gráfico: {e}")

# Gráfico: Dia da semana
print("\n📊 GRÁFICO: Volume por Dia da Semana")
print("   → Tipo: Barras verticais")
print("   → Eixo X: Dia da semana")
print("   → Eixo Y: Número médio de prescrições")
print("   → Insight: Identifica padrões semanais, ajudando no planejamento de recursos médicos")
try:
    fig_dow = px.bar(df_dow, x='dia_semana', y='prescricoes',
                     title='Volume de Prescrições por Dia da Semana',
                     labels={'dia_semana': 'Dia da Semana', 'prescricoes': 'Prescrições'},
                     template='plotly_white', color='prescricoes', color_continuous_scale='Blues')
    fig_dow.write_html('/tmp/prescricoes_dow.html')
    print("✓ Gráfico salvo: /tmp/prescricoes_dow.html")
    fig_dow.show()
except Exception as e:
    print(f"⚠️ Erro ao renderizar gráfico: {e}")

print("\n🎯 INSIGHTS Q1:")
print("-" * 60)
print(f"• Total: {stats_summary['Total']:,} prescrições em {stats_summary['Dias']} dias")
print(f"• Média diária: {stats_summary['Média Diária']} prescrições/dia")
print(f"• Variação: {stats_summary['Min']} (mín) a {stats_summary['Max']} (máx)")
print(f"• Dia com MAIOR volume: {df_dow.loc[df_dow['prescricoes'].idxmax(), 'dia_semana']}")
print(f"• Dia com MENOR volume: {df_dow.loc[df_dow['prescricoes'].idxmin(), 'dia_semana']}")
print("⚠️ ATENÇÃO: Padrão semanal identificado - importante para planejamento de recursos")

📊 ESTATÍSTICAS DE PRESCRIÇÕES DIÁRIAS
Total               :     70,907
Dias                :        108
Média Diária        :      656.5
Min                 :         35
Max                 :       1271

📅 SAZONALIDADE POR DIA DA SEMANA
dia_semana  num_dia  prescricoes
    Monday        0        12147
   Tuesday        1        12477
 Wednesday        2        12638
  Thursday        3        13071
    Friday        4        10720
  Saturday        5         5662
    Sunday     <NA>         4194

📈 GRÁFICO: Tendência Diária de Prescrições
   → Tipo: Linha temporal
   → Eixo X: Data da prescrição
   → Eixo Y: Número de prescrições por dia
   → Insight: Mostra a variação diária do volume, útil para identificar sazonalidade e picos de demanda
✓ Gráfico salvo: /tmp/prescrições_diárias.html



📊 GRÁFICO: Volume por Dia da Semana
   → Tipo: Barras verticais
   → Eixo X: Dia da semana
   → Eixo Y: Número médio de prescrições
   → Insight: Identifica padrões semanais, ajudando no planejamento de recursos médicos
✓ Gráfico salvo: /tmp/prescricoes_dow.html



🎯 INSIGHTS Q1:
------------------------------------------------------------
• Total: 70,907 prescrições em 108 dias
• Média diária: 656.5 prescrições/dia
• Variação: 35 (mín) a 1271 (máx)
• Dia com MAIOR volume: Thursday
• Dia com MENOR volume: Sunday
⚠️ ATENÇÃO: Padrão semanal identificado - importante para planejamento de recursos


---
## Q2: Quantos pacientes foram atendidos?

**O que analisamos:**
- Total de pacientes únicos
- Pacientes por sexo
- Distribuição geográfica (top estados)
- Retenção (quantas vezes cada paciente retorna)

**Por que importa:**
Retenção de pacientes é KPI crítico para crescimento sustentável da plataforma.

In [14]:
# Q2.1: Total de pacientes únicos
pacientes_total = con.execute("""
SELECT
    COUNT(DISTINCT idpaciente) AS pacientes_unicos,
    COUNT(DISTINCT idprescricao) AS prescricoes_unicas,
    ROUND(COUNT(DISTINCT idprescricao) * 1.0 / COUNT(DISTINCT idpaciente), 2) AS prescricoes_por_paciente
FROM prescricao
""").df()

print("👥 PACIENTES ATENDIDOS")
print("=" * 60)
print(pacientes_total.to_string(index=False))

# Q2.2: Distribuição por sexo
df_sexo = con.execute("""
SELECT
    CASE 
        WHEN UPPER(sexopaciente) IN ('F', 'FEMININO') THEN 'Feminino'
        WHEN UPPER(sexopaciente) IN ('M', 'MASCULINO') THEN 'Masculino'
        ELSE 'Não Informado'
    END AS sexo,
    COUNT(DISTINCT idpaciente) AS pacientes,
    ROUND(COUNT(DISTINCT idpaciente) * 100.0 / SUM(COUNT(DISTINCT idpaciente)) OVER (), 1) AS pct
FROM prescricao
GROUP BY 1
ORDER BY 2 DESC
""").df()

print("\n⚧ DISTRIBUIÇÃO POR SEXO")
print("=" * 60)
print(df_sexo.to_string(index=False))

# Q2.3: Top 10 estados
df_estado = con.execute("""
SELECT
    upper(estadopaciente) AS estado,
    COUNT(DISTINCT idpaciente) AS pacientes,
    ROUND(COUNT(DISTINCT idpaciente) * 100.0 / SUM(COUNT(DISTINCT idpaciente)) OVER (), 1) AS pct
FROM prescricao
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""").df()

print("\n🗺️  TOP 10 ESTADOS (pacientes)")
print("=" * 60)
print(df_estado.to_string(index=False))

# Q2.4: Retenção (quantas prescrições por paciente)
df_retencao = con.execute("""
WITH contagem AS (
    SELECT idpaciente, COUNT(DISTINCT idprescricao) AS num_prescricoes
    FROM prescricao
    GROUP BY 1
)
SELECT
    CASE
        WHEN num_prescricoes = 1 THEN '1 (Churn Risk)'
        WHEN num_prescricoes BETWEEN 2 AND 3 THEN '2-3 (Returning)'
        WHEN num_prescricoes BETWEEN 4 AND 10 THEN '4-10 (Engaged)'
        WHEN num_prescricoes > 10 THEN '11+ (Loyal)'
    END AS faixa,
    COUNT(*) AS pacientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_pacientes
FROM contagem
GROUP BY 1
ORDER BY CASE 
    WHEN faixa = '1 (Churn Risk)' THEN 1
    WHEN faixa = '2-3 (Returning)' THEN 2
    WHEN faixa = '4-10' THEN 3
    WHEN faixa = '11+ (Loyal)' THEN 4
END
""").df()

print("\n🔄 RETENÇÃO - PRESCRIÇÕES POR PACIENTE")
print("=" * 60)
print(df_retencao.to_string(index=False))

# Gráficos
print("\n📊 GRÁFICO: Distribuição de Pacientes por Sexo")
print("   → Tipo: Pizza")
print("   → Dados: Percentual de pacientes feminino vs masculino")
print("   → Insight: Mostra a demografia de gênero dos pacientes atendidos")
try:
    fig_sexo = px.pie(df_sexo, values='pacientes', names='sexo',
                      title='Distribuição de Pacientes por Sexo',
                      template='plotly_white')
    fig_sexo.write_html('/tmp/pacientes_sexo.html')
    print("\n✓ Gráfico salvo: /tmp/pacientes_sexo.html")
    fig_sexo.show()
except Exception as e:
    print(f"\n⚠️ Erro ao renderizar gráfico: {e}")

print("\n📊 GRÁFICO: Top Estados por Pacientes")
print("   → Tipo: Barras horizontais")
print("   → Eixo X: Número de pacientes")
print("   → Eixo Y: Estados")
print("   → Insight: Identifica os mercados regionais mais importantes")
try:
    fig_estado = px.bar(df_estado, x='estado', y='pacientes',
                        title='Top 10 Estados por Pacientes',
                        labels={'estado': 'Estado', 'pacientes': 'Pacientes'},
                        template='plotly_white', color='pct')
    fig_estado.write_html('/tmp/pacientes_estado.html')
    print("✓ Gráfico salvo: /tmp/pacientes_estado.html")
    fig_estado.show()
except Exception as e:
    print(f"⚠️ Erro ao renderizar gráfico: {e}")

print("\n📊 GRÁFICO: Distribuição de Retenção de Pacientes")
print("   → Tipo: Barras verticais")
print("   → Eixo X: Faixa de prescrições por paciente")
print("   → Eixo Y: Percentual de pacientes")
print("   → Insight: Mostra o nível de fidelização dos pacientes, crítico para crescimento sustentável")
try:
    fig_retencao = px.bar(df_retencao, x='faixa', y='pct_pacientes',
                          title='Distribuição de Retenção de Pacientes',
                          labels={'faixa': 'Prescrições por Paciente', 'pct_pacientes': '% Pacientes'},
                          template='plotly_white', color='pct_pacientes', color_continuous_scale='RdYlGn')
    fig_retencao.write_html('/tmp/pacientes_retencao.html')
    print("✓ Gráfico salvo: /tmp/pacientes_retencao.html")
    fig_retencao.show()
except Exception as e:
    print(f"⚠️ Erro ao renderizar gráfico: {e}")

print("\n🎯 INSIGHTS Q2:")
print("-" * 60)
total_pacs = pacientes_total['pacientes_unicos'].values[0]
churn_pct = df_retencao[df_retencao['faixa'] == '1 (Churn Risk)']['pct_pacientes'].values[0] if '1 (Churn Risk)' in df_retencao['faixa'].values else 0
return_pct = df_retencao[df_retencao['faixa'] == '2-3 (Returning)']['pct_pacientes'].values[0] if '2-3 (Returning)' in df_retencao['faixa'].values else 0
engage_pct = df_retencao[df_retencao['faixa'] == '4-10 (Engaged)']['pct_pacientes'].values[0] if '4-10 (Engaged)' in df_retencao['faixa'].values else 0
loyal_pct = df_retencao[df_retencao['faixa'] == '11+ (Loyal)']['pct_pacientes'].values[0] if '11+ (Loyal)' in df_retencao['faixa'].values else 0
print(f"• Total de pacientes: {total_pacs:,}")
print(f"• Média de prescrições/paciente: {pacientes_total['prescricoes_por_paciente'].values[0]}")
print(f"• ⚠️ {churn_pct}% dos pacientes têm apenas 1 prescrição (CHURN RISK)")
print(f"• ✅ {return_pct}% dos pacientes têm 2-3 prescrições (RETURNING)")
print(f"• ✅ {engage_pct}% dos pacientes têm 4-10 prescrições (ENGAGED)")
print(f"• ✅ {loyal_pct}% dos pacientes têm 11+ prescrições (LOYAL)")
print(f"• Estado com maior volume: {df_estado.iloc[0]['estado']} ({df_estado.iloc[0]['pacientes']:,} pacientes)")
print("⚠️ ATENÇÃO: ALTO CHURN - Programa de retenção é CRÍTICO")

👥 PACIENTES ATENDIDOS
 pacientes_unicos  prescricoes_unicas  prescricoes_por_paciente
            68767               70907                      1.03

⚧ DISTRIBUIÇÃO POR SEXO
         sexo  pacientes  pct
     Feminino      40273 58.6
    Masculino      25168 36.6
Não Informado       3326  4.8

🗺️  TOP 10 ESTADOS (pacientes)
estado  pacientes  pct
    SP      33570 48.8
   NaN       7575 11.0
    ES       3949  5.7
    PR       3880  5.6
    MG       3717  5.4
    RS       3401  4.9
    RJ       3185  4.6
    SC       3078  4.5
    RN       1326  1.9
    DF        975  1.4

🔄 RETENÇÃO - PRESCRIÇÕES POR PACIENTE
          faixa  pacientes  pct_pacientes
 1 (Churn Risk)      66749           97.1
2-3 (Returning)       2006            2.9
 4-10 (Engaged)         12            0.0

📊 GRÁFICO: Distribuição de Pacientes por Sexo
   → Tipo: Pizza
   → Dados: Percentual de pacientes feminino vs masculino
   → Insight: Mostra a demografia de gênero dos pacientes atendidos

✓ Gráfico salvo: /tmp/


📊 GRÁFICO: Top Estados por Pacientes
   → Tipo: Barras horizontais
   → Eixo X: Número de pacientes
   → Eixo Y: Estados
   → Insight: Identifica os mercados regionais mais importantes
✓ Gráfico salvo: /tmp/pacientes_estado.html



📊 GRÁFICO: Distribuição de Retenção de Pacientes
   → Tipo: Barras verticais
   → Eixo X: Faixa de prescrições por paciente
   → Eixo Y: Percentual de pacientes
   → Insight: Mostra o nível de fidelização dos pacientes, crítico para crescimento sustentável
✓ Gráfico salvo: /tmp/pacientes_retencao.html



🎯 INSIGHTS Q2:
------------------------------------------------------------
• Total de pacientes: 68,767
• Média de prescrições/paciente: 1.03
• ⚠️ 97.1% dos pacientes têm apenas 1 prescrição (CHURN RISK)
• ✅ 2.9% dos pacientes têm 2-3 prescrições (RETURNING)
• ✅ 0.0% dos pacientes têm 4-10 prescrições (ENGAGED)
• ✅ 0% dos pacientes têm 11+ prescrições (LOYAL)
• Estado com maior volume: SP (33,570 pacientes)
⚠️ ATENÇÃO: ALTO CHURN - Programa de retenção é CRÍTICO


---
## Q3: Quais especialidades médicas mais prescreveram?

**O que analisamos:**
- Ranking de especialidades por volume
- Open rate e conversão por especialidade
- Perfil dos médicos (gênero, idade, distribuição)

**Por que importa:**
Especialidades com baixo engajamento/conversão precisam de suporte diferenciado.

In [6]:
# Q3.1: Ranking de especialidades
df_esp = con.execute("""
SELECT
    m.especialidade,
    COUNT(DISTINCT p.idprescricao) AS prescricoes,
    COUNT(DISTINCT p.idmedico) AS medicos_ativos,
    COUNT(DISTINCT p.idpaciente) AS pacientes_unicos,
    ROUND(COUNT(DISTINCT p.idprescricao) * 100.0 / SUM(COUNT(DISTINCT p.idprescricao)) OVER (), 1) AS pct
FROM prescricao p
LEFT JOIN medicos m ON p.idmedico = m.idmedico
GROUP BY 1
ORDER BY 2 DESC
""").df()

print("🏥 TOP ESPECIALIDADES (Volume de Prescrições)")
print("=" * 80)
print(df_esp.to_string(index=False))

# Q3.2: Open Rate e Conversão por especialidade
df_esp_kpi = con.execute("""
WITH presc_nivel AS (
    SELECT
        p.idprescricao,
        m.especialidade,
        MAX(p.visualizadapaciente::INT) AS visualizada,
        MAX(CASE WHEN p.itemvendido > 0 THEN 1 ELSE 0 END) AS vendida
    FROM prescricao p
    LEFT JOIN medicos m ON p.idmedico = m.idmedico
    GROUP BY 1, 2
)
SELECT
    especialidade,
    COUNT(*) AS prescricoes,
    SUM(visualizada) AS visualizadas,
    SUM(vendida) AS vendidas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 1) AS open_rate_pct,
    ROUND(SUM(vendida) * 100.0 / NULLIF(SUM(visualizada), 0), 1) AS conversao_pct
FROM presc_nivel
GROUP BY 1
HAVING COUNT(*) >= 30
ORDER BY 2 DESC
""").df()

print("\n📊 PERFORMANCE POR ESPECIALIDADE (Open Rate + Conversão)")
print("=" * 80)
print(df_esp_kpi.to_string(index=False))

# Q3.3: Perfil dos médicos
df_medicos_perfil = con.execute("""
SELECT
    genero,
    COUNT(DISTINCT idmedico) AS medicos,
    ROUND(AVG(idade), 1) AS idade_media,
    MIN(idade) AS idade_min,
    MAX(idade) AS idade_max
FROM medicos
WHERE idade IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC
""").df()

print("\n👨‍⚕️  PERFIL DOS MÉDICOS")
print("=" * 80)
print(df_medicos_perfil.to_string(index=False))

# Q3.4: Estados com maior concentração de médicos
df_medicos_estado = con.execute("""
SELECT
    estado,
    COUNT(DISTINCT idmedico) AS medicos,
    ROUND(COUNT(DISTINCT idmedico) * 100.0 / SUM(COUNT(DISTINCT idmedico)) OVER (), 1) AS pct
FROM medicos
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""").df()

print("\n🗺️  TOP 10 ESTADOS (Médicos)")
print("=" * 80)
print(df_medicos_estado.to_string(index=False))

# Gráficos
print("\n📊 GRÁFICO: Top Especialidades por Volume")
print("   → Tipo: Barras horizontais")
print("   → Eixo X: Número de prescrições")
print("   → Eixo Y: Especialidades médicas")
print("   → Insight: Mostra quais especialidades geram mais prescrições digitais")
fig_esp = px.bar(df_esp.head(15), x='prescricoes', y='especialidade', orientation='h',
                 title='Top 15 Especialidades por Volume',
                 labels={'prescricoes': 'Prescrições', 'especialidade': 'Especialidade'},
                 template='plotly_white', color='pct', color_continuous_scale='Blues')
fig_esp.show()

print("\n📊 GRÁFICO: Open Rate vs Conversão por Especialidade")
print("   → Tipo: Scatter plot (dispersão)")
print("   → Eixo X: Taxa de abertura (%)")
print("   → Eixo Y: Taxa de conversão (%)")
print("   → Bolhas: Tamanho proporcional ao volume de prescrições")
print("   → Insight: Identifica especialidades com baixo engajamento que precisam de suporte")
fig_esp_kpi = px.scatter(df_esp_kpi, x='open_rate_pct', y='conversao_pct', 
                        size='prescricoes', hover_name='especialidade',
                        title='Open Rate vs Conversão por Especialidade',
                        labels={'open_rate_pct': 'Open Rate (%)', 'conversao_pct': 'Conversão (%)'},
                        template='plotly_white', color='prescricoes', color_continuous_scale='Viridis')
fig_esp_kpi.show()

print("\n🎯 INSIGHTS Q3:")
print("-" * 80)
top_esp = df_esp.iloc[0]
low_perform = df_esp_kpi[df_esp_kpi['open_rate_pct'] < 50]
print(f"• Especialidade TOP: {top_esp['especialidade']} ({top_esp['prescricoes']:,} prescrições)")
print(f"• {len(df_esp)} especialidades cadastradas")
if len(low_perform) > 0:
    print(f"• ⚠️ {len(low_perform)} especialidades com Open Rate < 50%:")
    for _, row in low_perform.head(3).iterrows():
        print(f"   - {row['especialidade']}: {row['open_rate_pct']}% (Conversão: {row['conversao_pct']}%)")
print("✅ RECOMENDAÇÃO: Focar em especialidades com baixo engajamento para suporte")

🏥 TOP ESPECIALIDADES (Volume de Prescrições)
                            especialidade  prescricoes  medicos_ativos  pacientes_unicos  pct
                           CLINICA MEDICA        15293            3083             15035 21.6
                        SEM ESPECIALIDADE        12959            2457             12668 18.3
                                PEDIATRIA         8702            1968              8471 12.3
                           CIRURGIA GERAL         5380             958              5307  7.6
                ORTOPEDIA E TRAUMATOLOGIA         4473             947              4420  6.3
         MEDICINA DE FAMILIA E COMUNIDADE         2804             530              2734  4.0
                GINECOLOGIA E OBSTETRICIA         2687             999              2634  3.8
                              PSIQUIATRIA         2633             507              2446  3.7
             ENDOCRINOLOGIA E METABOLOGIA         2088             292              2034  2.9
               


🎯 INSIGHTS Q3:
--------------------------------------------------------------------------------
• Especialidade TOP: CLINICA MEDICA (15,293 prescrições)
• 93 especialidades cadastradas
• ⚠️ 33 especialidades com Open Rate < 50%:
   - CLINICA MEDICA: 43.2% (Conversão: 13.2%)
   - PEDIATRIA: 47.6% (Conversão: 9.2%)
   - ORTOPEDIA E TRAUMATOLOGIA: 41.3% (Conversão: 18.2%)
✅ RECOMENDAÇÃO: Focar em especialidades com baixo engajamento para suporte


---
## Q4: Qual é a taxa de Open Rate?

**O que analisamos:**
- Taxa geral de abertura de receitas
- Open rate por dia da semana
- Open rate por hora de emissão
- Open rate por estado

**Por que importa:**
Open rate baixo = oportunidade de notificação push (SMS/WhatsApp) para aumentar conversão.

In [15]:
# Q4.1: Open Rate geral
df_or_geral = con.execute("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    COUNT(*) AS total_prescricoes,
    SUM(visualizada) AS visualizadas,
    COUNT(*) - SUM(visualizada) AS nao_visualizadas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 2) AS open_rate_pct
FROM presc_nivel
""").df()

print("📊 OPEN RATE GERAL")
print("=" * 60)
print(df_or_geral.to_string(index=False))

or_geral = df_or_geral['open_rate_pct'].values[0]
not_opened = df_or_geral['nao_visualizadas'].values[0]

print(f"\n💡 De {df_or_geral['total_prescricoes'].values[0]:,} prescrições, {not_opened:,} ({100-or_geral:.1f}%) NÃO foram abertas!")

# Q4.2: Open Rate por dia da semana
df_or_dow = con.execute("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        DAYNAME(MIN(dataprescricao::TIMESTAMP)) AS dia_semana,
        DAYOFWEEK(MIN(dataprescricao::TIMESTAMP)) AS num_dia,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    dia_semana,
    COUNT(*) AS prescricoes,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct
FROM presc_nivel
GROUP BY 1, num_dia
ORDER BY num_dia
""").df()

print("\n📅 OPEN RATE POR DIA DA SEMANA")
print("=" * 60)
print(df_or_dow.to_string(index=False))

# Q4.3: Open Rate por hora
df_or_hora = con.execute("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        HOUR(MIN(dataprescricao::TIMESTAMP)) AS hora,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    hora,
    COUNT(*) AS prescricoes,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct
FROM presc_nivel
GROUP BY 1
ORDER BY 1
""").df()

print("\n🕐 OPEN RATE POR HORA DE EMISSÃO")
print("=" * 60)
print(df_or_hora.to_string(index=False))

# Q4.4: Open Rate por estado (problema de cobertura)
df_or_estado = con.execute("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(upper(estadopaciente)) AS estado,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    estado,
    COUNT(*) AS prescricoes,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct
FROM presc_nivel
GROUP BY 1
HAVING COUNT(*) >= 30
ORDER BY open_rate_pct ASC
LIMIT 15
""").df()

print("\n🗺️  ESTADOS COM MAIOR PROBLEMA DE ABERTURA (Open Rate Baixo)")
print("=" * 60)
print(df_or_estado.to_string(index=False))

# Gráficos
print("\n📊 GRÁFICO: Open Rate por Dia da Semana")
print("   → Tipo: Barras verticais")
print("   → Eixo X: Dia da semana")
print("   → Eixo Y: Taxa de abertura (%)")
print("   → Insight: Mostra quando as receitas são mais visualizadas pelos pacientes")
fig_or_dow = px.bar(df_or_dow, x='dia_semana', y='open_rate_pct',
                    title='Open Rate por Dia da Semana',
                    labels={'dia_semana': 'Dia', 'open_rate_pct': 'Open Rate (%)'},
                    template='plotly_white', color='open_rate_pct', color_continuous_scale='RdYlGn')
fig_or_dow.show()

print("\n📊 GRÁFICO: Open Rate por Hora do Dia")
print("   → Tipo: Linha")
print("   → Eixo X: Hora da emissão")
print("   → Eixo Y: Taxa de abertura (%)")
print("   → Insight: Identifica horários ideais para notificações push")
fig_or_hora = px.line(df_or_hora, x='hora', y='open_rate_pct', markers=True,
                      title='Open Rate por Hora do Dia',
                      labels={'hora': 'Hora', 'open_rate_pct': 'Open Rate (%)'},
                      template='plotly_white')
fig_or_hora.show()

print("\n📊 GRÁFICO: Estados com Menor Open Rate")
print("   → Tipo: Barras horizontais")
print("   → Eixo X: Estado")
print("   → Eixo Y: Taxa de abertura (%)")
print("   → Insight: Estados que precisam de campanhas de ativação regional")
fig_or_estado = px.bar(df_or_estado, x='estado', y='open_rate_pct',
                       title='Estados com Menor Open Rate (Oportunidade de Ativação)',
                       labels={'estado': 'Estado', 'open_rate_pct': 'Open Rate (%)'},
                       template='plotly_white', color='open_rate_pct', color_continuous_scale='Reds')
fig_or_estado.show()

print("\n🎯 INSIGHTS Q4:")
print("-" * 60)
print(f"• Open Rate Geral: {or_geral:.1f}%")
print(f"• Receitas NÃO abertas: {not_opened:,} ({100-or_geral:.1f}%)")
best_dow = df_or_dow.loc[df_or_dow['open_rate_pct'].idxmax()]
worst_dow = df_or_dow.loc[df_or_dow['open_rate_pct'].idxmin()]
print(f"• Melhor dia: {best_dow['dia_semana']} ({best_dow['open_rate_pct']}%)")
print(f"• Pior dia: {worst_dow['dia_semana']} ({worst_dow['open_rate_pct']}%)")
print(f"• ⚠️ Estados críticos (< 45% open rate): {len(df_or_estado[df_or_estado['open_rate_pct'] < 45])}")
print("\n🚀 ATIVAÇÃO IMEDIATA:")
print("   → Implementar notificação push (SMS/WhatsApp) para receitas não abertas")
print("   → Foco em estados com baixo open rate + horários de pico")
print(f"   → Potencial de adicionar +{(not_opened * 0.5):.0f} prescrições vizualizadas com notificação")

📊 OPEN RATE GERAL
 total_prescricoes  visualizadas  nao_visualizadas  open_rate_pct
             70907       35772.0           35135.0          50.45

💡 De 70,907 prescrições, 35,135.0 (49.5%) NÃO foram abertas!

📅 OPEN RATE POR DIA DA SEMANA
dia_semana  prescricoes  open_rate_pct
    Sunday         4194           48.6
    Monday        12147           51.8
   Tuesday        12477           50.6
 Wednesday        12638           50.5
  Thursday        13071           50.1
    Friday        10719           49.6
  Saturday         5661           50.8

🕐 OPEN RATE POR HORA DE EMISSÃO
 hora  prescricoes  open_rate_pct
    0         1893           49.2
    1         1331           45.8
    2         1126           42.8
    3          749           41.5
    4          501           41.7
    5          357           37.0
    6          265           37.4
    7          225           39.1
    8          280           51.1
    9          577           61.5
   10         2165           58.7
   1


📊 GRÁFICO: Open Rate por Hora do Dia
   → Tipo: Linha
   → Eixo X: Hora da emissão
   → Eixo Y: Taxa de abertura (%)
   → Insight: Identifica horários ideais para notificações push



📊 GRÁFICO: Estados com Menor Open Rate
   → Tipo: Barras horizontais
   → Eixo X: Estado
   → Eixo Y: Taxa de abertura (%)
   → Insight: Estados que precisam de campanhas de ativação regional



🎯 INSIGHTS Q4:
------------------------------------------------------------
• Open Rate Geral: 50.5%
• Receitas NÃO abertas: 35,135.0 (49.5%)
• Melhor dia: Monday (51.8%)
• Pior dia: Sunday (48.6%)
• ⚠️ Estados críticos (< 45% open rate): 7

🚀 ATIVAÇÃO IMEDIATA:
   → Implementar notificação push (SMS/WhatsApp) para receitas não abertas
   → Foco em estados com baixo open rate + horários de pico
   → Potencial de adicionar +17568 prescrições vizualizadas com notificação


---
## Q5: Qual é a taxa de conversão por canal?

**O que analisamos:**
- Funil completo (emitidas → abertas → vendidas)
- Conversão por canal (farmácia física vs marketplace)
- Distribuição de vendas por canal

**Por que importa:**
Canal marketplace precisa melhorar a taxa de conversao.

In [16]:
# Q5.1: Funil completo
df_funnel = con.execute("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT) AS visualizada,
        MAX(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) AS vendida
    FROM prescricao
    GROUP BY 1
)
SELECT
    COUNT(*) AS emitidas,
    SUM(visualizada) AS visualizadas,
    SUM(vendida) AS vendidas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 1) AS open_rate_pct,
    ROUND(SUM(vendida) * 100.0 / NULLIF(SUM(visualizada), 0), 1) AS conversao_pct_visualizadas,
    ROUND(SUM(vendida) * 100.0 / COUNT(*), 1) AS conversao_pct_total
FROM presc_nivel
""").df()

print("📊 FUNIL COMPLETO (Emitidas → Abertas → Vendidas)")
print("=" * 80)
print(df_funnel.to_string(index=False))

# Q5.2: Conversão por canal
df_canal = con.execute("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT) AS visualizada,
        MAX(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) AS vendida,
        COALESCE(
            MAX(CASE WHEN canalvenda != 'não convertido' THEN canalvenda END),
            'Não Convertido'
        ) AS canal_final
    FROM prescricao
    GROUP BY 1
)
SELECT
    canal_final,
    COUNT(*) AS prescricoes,
    SUM(visualizada) AS visualizadas,
    SUM(vendida) AS vendidas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 1) AS open_rate_pct,
    ROUND(SUM(vendida) * 100.0 / NULLIF(SUM(visualizada), 0), 1) AS conversao_pct
FROM presc_nivel
GROUP BY 1
ORDER BY 3 DESC
""").df()

print("\n🏪 CONVERSÃO POR CANAL")
print("=" * 80)
print(df_canal.to_string(index=False))

# Q5.3: Tempo até conversão (comparação marketplace vs farmacia)
df_tempo_canal = con.execute("""
WITH vendas AS (
    SELECT
        idprescricao,
        COALESCE(
            MAX(CASE WHEN canalvenda != 'não convertido' THEN canalvenda END),
            'Não Convertido'
        ) AS canal,
        DATEDIFF('hour', MIN(dataprescricao::TIMESTAMP), MIN(datavenda::TIMESTAMP)) AS horas
    FROM prescricao
    WHERE datavenda IS NOT NULL
    GROUP BY 1
    HAVING DATEDIFF('hour', MIN(dataprescricao::TIMESTAMP), MIN(datavenda::TIMESTAMP)) >= 0
)
SELECT
    canal,
    COUNT(*) AS vendas,
    ROUND(AVG(horas), 1) AS media_horas,
    ROUND(AVG(horas)/24, 1) AS media_dias,
    ROUND(MEDIAN(horas), 1) AS mediana_horas
FROM vendas
WHERE canal != 'Não Convertido'
GROUP BY 1
ORDER BY 2 DESC
""").df()

print("\n⏱️  TEMPO MÉDIO ATÉ CONVERSÃO POR CANAL")
print("=" * 80)
print(df_tempo_canal.to_string(index=False))

# Gráficos
print("\n📊 GRÁFICO: Funil de Conversão Geral")
print("   → Tipo: Funil")
print("   → Etapas: Prescrições Emitidas → Receitas Abertas → Vendas Realizadas")
print("   → Insight: Mostra as perdas em cada etapa do processo de conversão")
fig_funnel = go.Figure(go.Funnel(
    y=["Prescrições Emitidas", "Receitas Abertas", "Vendas Realizadas"],
    x=[df_funnel['emitidas'].values[0], df_funnel['visualizadas'].values[0], df_funnel['vendidas'].values[0]],
    textinfo="value+percent initial",
    marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
))
fig_funnel.update_layout(
    title="Funil de Conversão Geral",
    height=400,
    template='plotly_white'
)
fig_funnel.show()

print("\n📊 GRÁFICO: Prescrições Abertas vs Vendidas por Canal")
print("   → Tipo: Barras agrupadas")
print("   → Eixo X: Canal de venda")
print("   → Barras: Abertas (azul) vs Vendidas (laranja)")
print("   → Insight: Compara performance entre marketplace e farmácia física")
fig_canal = px.bar(df_canal, x='canal_final', y=['visualizadas', 'vendidas'],
                   title='Prescrições Abertas vs Vendidas por Canal',
                   labels={'value': 'Quantidade'},
                   barmode='group',
                   template='plotly_white')
fig_canal.show()

print("\n📊 GRÁFICO: Tempo Médio até Conversão por Canal")
print("   → Tipo: Barras verticais")
print("   → Eixo X: Canal")
print("   → Eixo Y: Tempo médio em horas")
print("   → Insight: Mostra velocidade de conversão - quanto mais rápido, melhor experiência")
fig_tempo = px.bar(df_tempo_canal, x='canal', y='media_horas',
                   title='Tempo Médio até Conversão por Canal',
                   labels={'canal': 'Canal', 'media_horas': 'Horas'},
                   template='plotly_white', color='media_horas', color_continuous_scale='Reds_r')
fig_tempo.show()

print("\n🎯 INSIGHTS Q5:")
print("-" * 80)
emitidas = df_funnel['emitidas'].values[0]
conv_total = df_funnel['conversao_pct_total'].values[0]
conv_vis = df_funnel['conversao_pct_visualizadas'].values[0]
print(f"• Funil: {emitidas:,} emitidas → {df_funnel['visualizadas'].values[0]:,} abertas → {df_funnel['vendidas'].values[0]:,} vendidas")
print(f"• Conversão de visualizadas → vendidas: {conv_vis}%")
print(f"• Conversão total (emitidas → vendidas): {conv_total}%")

marketplace_row = df_canal[df_canal['canal_final'] == 'marketplace']
farmacia_row = df_canal[df_canal['canal_final'] == 'farmacia fisica']

if not marketplace_row.empty:
    mp_conv = marketplace_row['conversao_pct'].values[0]
    print(f"• Marketplace: {mp_conv}% conversão")
if not farmacia_row.empty:
    ff_conv = farmacia_row['conversao_pct'].values[0]
    print(f"• Farmácia Física: {ff_conv}% conversão")

if not df_tempo_canal.empty:
    print(f"\n⏱️  VELOCIDADE DE CONVERSÃO:")
    for _, row in df_tempo_canal.iterrows():
        print(f"   - {row['canal']}: {row['media_horas']}h ({row['media_dias']} dias)")

print("\n✅ RECOMENDAÇÃO: Melhorar notificacoes para o usuario lembrar de abrir a receita e saber que tem disponibilidade de compra online, focando em canais com menor tempo de conversão")

📊 FUNIL COMPLETO (Emitidas → Abertas → Vendidas)
 emitidas  visualizadas  vendidas  open_rate_pct  conversao_pct_visualizadas  conversao_pct_total
    70907       35772.0    4601.0           50.4                        12.9                  6.5

🏪 CONVERSÃO POR CANAL
    canal_final  prescricoes  visualizadas  vendidas  open_rate_pct  conversao_pct
 Não Convertido        66306       32141.0       0.0           48.5            0.0
farmacia fisica         3873        2903.0    3873.0           75.0          133.4
    marketplace          728         728.0     728.0          100.0          100.0

⏱️  TEMPO MÉDIO ATÉ CONVERSÃO POR CANAL
          canal  vendas  media_horas  media_dias  mediana_horas
farmacia fisica    3865         80.5         3.4            9.0
    marketplace     728         57.7         2.4            4.0

📊 GRÁFICO: Funil de Conversão Geral
   → Tipo: Funil
   → Etapas: Prescrições Emitidas → Receitas Abertas → Vendas Realizadas
   → Insight: Mostra as perdas em cada e


📊 GRÁFICO: Prescrições Abertas vs Vendidas por Canal
   → Tipo: Barras agrupadas
   → Eixo X: Canal de venda
   → Barras: Abertas (azul) vs Vendidas (laranja)
   → Insight: Compara performance entre marketplace e farmácia física



📊 GRÁFICO: Tempo Médio até Conversão por Canal
   → Tipo: Barras verticais
   → Eixo X: Canal
   → Eixo Y: Tempo médio em horas
   → Insight: Mostra velocidade de conversão - quanto mais rápido, melhor experiência



🎯 INSIGHTS Q5:
--------------------------------------------------------------------------------
• Funil: 70,907 emitidas → 35,772.0 abertas → 4,601.0 vendidas
• Conversão de visualizadas → vendidas: 12.9%
• Conversão total (emitidas → vendidas): 6.5%
• Marketplace: 100.0% conversão
• Farmácia Física: 133.4% conversão

⏱️  VELOCIDADE DE CONVERSÃO:
   - farmacia fisica: 80.5h (3.4 dias)
   - marketplace: 57.7h (2.4 dias)

✅ RECOMENDAÇÃO: Melhorar notificacoes para o usuario lembrar de abrir a receita e saber que tem disponibilidade de compra online, focando em canais com menor tempo de conversão


---
## Q6: Que outras informações podemos visualizar?

**O que analisamos:**
- Top medicamentos mais prescritos
- Medicamentos controlados vs antimicrobianos
- Taxa de conversão por medicamento
- Faixa etária dos pacientes (idade)

**Por que importa:**
Medicamentos com baixo engajamento podem ter problemas de disponibilidade nas farmácias parceiras.

In [22]:
# Q6.1: Top 20 medicamentos
df_med_top = con.execute("""
SELECT
    med.nome,
    COUNT(DISTINCT p.idprescricao) AS prescricoes,
    SUM(p.visualizadapaciente::INT) AS visualizadas,
    SUM(CASE WHEN p.itemvendido > 0 THEN 1 ELSE 0 END) AS vendidas,
    ROUND(SUM(CASE WHEN p.itemvendido > 0 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(p.visualizadapaciente::INT), 0), 1) AS conversao_pct,
    SUM(CASE WHEN p.canalvenda = 'marketplace' THEN 1 ELSE 0 END) AS vendas_marketplace,
    SUM(CASE WHEN p.canalvenda = 'farmacia fisica' THEN 1 ELSE 0 END) AS vendas_farmacia,
    ROUND(SUM(CASE WHEN p.canalvenda = 'marketplace' THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN p.itemvendido > 0 THEN 1 ELSE 0 END), 0), 1) AS pct_marketplace,
    ROUND(SUM(CASE WHEN p.canalvenda = 'farmacia fisica' THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN p.itemvendido > 0 THEN 1 ELSE 0 END), 0), 1) AS pct_farmacia,
    med.antimicrobiano,
    med.controleespecial,
    med.mip
FROM prescricao p
LEFT JOIN medicamentos med ON p.idmedicamento = med.idmedicamento
GROUP BY 1, 10, 11, 12
ORDER BY 2 DESC
LIMIT 20
""").df()

print("💊 TOP 20 MEDICAMENTOS")
print("=" * 100)
print(df_med_top.to_string(index=False))

# Q6.2: Classificação de medicamentos
df_med_class = con.execute("""
SELECT
    CASE 
        WHEN antimicrobiano = true THEN 'Antimicrobiano'
        WHEN controleespecial = true THEN 'Controlado'
        WHEN mip = true THEN 'MIP'
        ELSE 'Comum'
    END AS classe,
    COUNT(DISTINCT p.idprescricao) AS prescricoes,
    ROUND(COUNT(DISTINCT p.idprescricao) * 100.0 / SUM(COUNT(DISTINCT p.idprescricao)) OVER (), 1) AS pct
FROM prescricao p
LEFT JOIN medicamentos med ON p.idmedicamento = med.idmedicamento
GROUP BY 1
ORDER BY 2 DESC
""").df()

print("\n🏷️  DISTRIBUIÇÃO POR CLASSE DE MEDICAMENTO")
print("=" * 100)
print(df_med_class.to_string(index=False))

# Q6.3: Idade dos pacientes (usando nascimento)
df_idade = con.execute("""
SELECT
    CASE
        WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) < 18 THEN '< 18'
        WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 18 AND 25 THEN '18-25'
        WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 26 AND 35 THEN '26-35'
        WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 36 AND 45 THEN '36-45'
        WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 46 AND 55 THEN '46-55'
        WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 56 AND 65 THEN '56-65'
        ELSE '65+'
    END AS faixa_etaria,
    COUNT(DISTINCT p.idprescricao) AS prescricoes,
    COUNT(DISTINCT p.idpaciente) AS pacientes,
    ROUND(COUNT(DISTINCT p.idprescricao) * 100.0 / SUM(COUNT(DISTINCT p.idprescricao)) OVER (), 1) AS pct
FROM prescricao p
WHERE TRY_CAST(nascimentopaciente AS DATE) IS NOT NULL
GROUP BY 1
ORDER BY prescricoes DESC
""").df()

print("\n👥 DISTRIBUIÇÃO POR FAIXA ETÁRIA")
print("=" * 100)
print(df_idade.to_string(index=False))

# Q6.4: Open Rate e Conversão por faixa etária
df_idade_kpi = con.execute("""
WITH presc_idade AS (
    SELECT
        CASE
            WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) < 18 THEN '< 18'
            WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 18 AND 25 THEN '18-25'
            WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 26 AND 35 THEN '26-35'
            WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 36 AND 45 THEN '36-45'
            WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 46 AND 55 THEN '46-55'
            WHEN YEAR(CURRENT_DATE) - YEAR(TRY_CAST(nascimentopaciente AS DATE)) BETWEEN 56 AND 65 THEN '56-65'
            ELSE '65+'
        END AS faixa_etaria,
        idprescricao,
        visualizadapaciente,
        itemvendido
    FROM prescricao
    WHERE TRY_CAST(nascimentopaciente AS DATE) IS NOT NULL
)
SELECT
    faixa_etaria,
    COUNT(DISTINCT idprescricao) AS prescricoes,
    ROUND(AVG(visualizadapaciente::INT) * 100, 1) AS open_rate_pct,
    ROUND(SUM(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) * 100.0 / SUM(CASE WHEN visualizadapaciente = true THEN 1 ELSE 0 END), 1) AS conversao_pct
FROM presc_idade
GROUP BY 1
ORDER BY prescricoes DESC
""").df()

print("\n📊 KPIs POR FAIXA ETÁRIA")
print("=" * 100)
print(df_idade_kpi.to_string(index=False))

# Gráficos
print("\n📊 GRÁFICO: Top Medicamentos por Volume")
print("   → Tipo: Barras horizontais")
print("   → Eixo X: Número de prescrições")
print("   → Eixo Y: Nome do medicamento")
print("   → Insight: Mostra os medicamentos mais prescritos digitalmente")
fig_med = px.bar(df_med_top.head(15), x='prescricoes', y='nome', orientation='h',
                 title='Top 15 Medicamentos',
                 labels={'prescricoes': 'Prescrições', 'nome': 'Medicamento'},
                 template='plotly_white', color='conversao_pct', color_continuous_scale='Blues')
fig_med.show()

print("\n📊 GRÁFICO: Distribuição por Classe de Medicamento")
print("   → Tipo: Pizza")
print("   → Fatias: Classes (Antimicrobiano, Controlado, MIP, Comum)")
print("   → Insight: Mostra a composição do portfólio de medicamentos prescritos")
fig_classe = px.pie(df_med_class, values='prescricoes', names='classe',
                    title='Distribuição de Medicamentos por Classe',
                    template='plotly_white')
fig_classe.show()

print("\n📊 GRÁFICO: Open Rate e Conversão por Faixa Etária")
print("   → Tipo: Barras agrupadas")
print("   → Eixo X: Faixa etária")
print("   → Barras: Open Rate vs Conversão")
print("   → Insight: Identifica perfis de pacientes com melhor engajamento")
fig_idade = px.bar(df_idade_kpi, x='faixa_etaria', y=['open_rate_pct', 'conversao_pct'],
                   title='Open Rate e Conversão por Faixa Etária',
                   barmode='group',
                   labels={'value': '%'},
                   template='plotly_white')
fig_idade.show()

print("\n🎯 INSIGHTS Q6:")
print("-" * 100)
top_med = df_med_top.iloc[0]
print(f"• Medicamento TOP: {top_med['nome']} ({top_med['prescricoes']:,} prescrições)")
print(f"• Medicamentos cadastrados: {len(df_med_top)}")
med_controled = df_med_class[df_med_class['classe'] == 'Controlado']['pct'].values[0] if 'Controlado' in df_med_class['classe'].values else 0
print(f"• Medicamentos controlados: {med_controled}% do volume")
print(f"• Faixa etária com maior volume: {df_idade.iloc[0]['faixa_etaria']} ({df_idade.iloc[0]['prescricoes']:,} prescr.)")
best_age = df_idade_kpi.loc[df_idade_kpi['open_rate_pct'].idxmax()]
print(f"• Faixa etária com melhor engajamento: {best_age['faixa_etaria']} ({best_age['open_rate_pct']}% open rate)")

💊 TOP 20 MEDICAMENTOS
                      nome  prescricoes  visualizadas  vendidas  conversao_pct  vendas_marketplace  vendas_farmacia  pct_marketplace  pct_farmacia  antimicrobiano  controleespecial   mip
               Paracetamol        12860        7086.0     388.0            5.5                50.0            338.0             12.9          87.1           False              True  True
                Glifage XR         8920        4056.0     408.0           10.1                61.0            347.0             15.0          85.0           False             False False
             Aerolin Spray         6484        3262.0     325.0           10.0                30.0            295.0              9.2          90.8           False             False False
                    Avamys         6250        3389.0     314.0            9.3                95.0            219.0             30.3          69.7           False             False False
     Sulfato de Salbutamol         5855    


📊 GRÁFICO: Distribuição por Classe de Medicamento
   → Tipo: Pizza
   → Fatias: Classes (Antimicrobiano, Controlado, MIP, Comum)
   → Insight: Mostra a composição do portfólio de medicamentos prescritos



📊 GRÁFICO: Open Rate e Conversão por Faixa Etária
   → Tipo: Barras agrupadas
   → Eixo X: Faixa etária
   → Barras: Open Rate vs Conversão
   → Insight: Identifica perfis de pacientes com melhor engajamento



🎯 INSIGHTS Q6:
----------------------------------------------------------------------------------------------------
• Medicamento TOP: Paracetamol (12,860 prescrições)
• Medicamentos cadastrados: 20
• Medicamentos controlados: 26.6% do volume
• Faixa etária com maior volume: 26-35 (12,854 prescr.)
• Faixa etária com melhor engajamento: 26-35 (59.1% open rate)


---
## Q7: Insights Operacionais & Recomendações Estratégicas

**Objetivo:**
Consolidar todos os dados em recomendações ACIONÁVEIS para melhorar experiência do paciente e eficiência operacional.

In [11]:
# Q7.1: Oportunidades de Remarketing (não abertas 6-24h)
not_opened_potential = con.execute("""
SELECT
    COUNT(DISTINCT idprescricao) AS prescricoes_elegíveis,
    ROUND(COUNT(DISTINCT idprescricao) * 0.5, 0) AS potencial_com_notificacao
FROM prescricao p
WHERE visualizadapaciente = false
""").df()

# Q7.2: Médicos em risco de churn
medicos_churn = con.execute("""
SELECT
    COUNT(DISTINCT idmedico) AS medicos_1_presc_apenas,
    ROUND(COUNT(DISTINCT idmedico) * 100.0 / (SELECT COUNT(DISTINCT idmedico) FROM prescricao), 1) AS pct
FROM (
    SELECT idmedico, COUNT(DISTINCT idprescricao) AS num
    FROM prescricao
    GROUP BY 1
    HAVING COUNT(DISTINCT idprescricao) = 1
)
""").df()

# Q7.3: Convênios
convenio_dist = con.execute("""
SELECT
    CASE WHEN idconvenio IS NULL THEN 'Particular' ELSE 'Convênio' END AS tipo,
    COUNT(DISTINCT idprescricao) AS prescricoes,
    ROUND(COUNT(DISTINCT idprescricao) * 100.0 / SUM(COUNT(DISTINCT idprescricao)) OVER (), 1) AS pct
FROM prescricao
GROUP BY 1
""").df()

# Q7.4: Retenção crítica
churn_1_presc = con.execute("""
SELECT COUNT(DISTINCT idpaciente) AS pacientes_apenas_1_presc
FROM (
    SELECT idpaciente, COUNT(DISTINCT idprescricao) AS num
    FROM prescricao
    GROUP BY 1
    HAVING COUNT(DISTINCT idprescricao) = 1
)
""").df()

print("=" * 100)
print("🎯 Q7 - INSIGHTS OPERACIONAIS & RECOMENDAÇÕES ESTRATÉGICAS")
print("=" * 100)

print("\n" + "█" * 100)
print("1️⃣  ATIVAÇÃO IMEDIATA: NOTIFICAÇÃO PUSH (Impact Alto | Effort Baixo)")
print("█" * 100)

not_opened = not_opened_potential['prescricoes_elegíveis'].values[0]
potential_conversion = not_opened_potential['potencial_com_notificacao'].values[0]

print(f"\n📱 PROBLEMA:")
print(f"   → {not_opened:,} prescrições ({(not_opened/df_funnel['emitidas'].values[0]*100):.1f}%) não foram ABERTAS")
print(f"   → Falta de notificação = perda de oportunidade de conversão")

print(f"\n✅ SOLUÇÃO:")
print(f"   → Implementar SMS/WhatsApp para receitas não abertas (6-24h após emissão)")
print(f"   → Personalizado por canal (marketplace vs farmácia física)")
print(f"   → A/B test com horários de envio")

print(f"\n💰 ROI ESPERADO:")
print(f"   → Se 50% das não-abertas virem a abrir: +{potential_conversion:.0f} visualizações")
print(f"   → Com conversão média de {df_funnel['conversao_pct_visualizadas'].values[0]}%, ganho: ~{(potential_conversion * df_funnel['conversao_pct_visualizadas'].values[0]/100):.0f} vendas adicionais")

print("\n" + "█" * 100)
print("2️⃣  RETENÇÃO CRÍTICA: PROGRAMA DE FIDELIZAÇÃO (Impact Crítico)")
print("█" * 100)

churn_pct = churn_1_presc['pacientes_apenas_1_presc'].values[0]
total_pacs_churn = pacientes_total['pacientes_unicos'].values[0]

print(f"\n⚠️ PROBLEMA:")
print(f"   → {churn_pct:,} pacientes ({(churn_pct/total_pacs_churn*100):.1f}%) com apenas 1 prescrição = CHURN ALTO")
print(f"   → Custo de aquisição não amortiza")

print(f"\n✅ SOLUÇÃO:")
print(f"   → Email marketing: oferecer desconto na 2ª prescrição")
print(f"   → Push notification: 'Sua receita expirou? Renove aqui'")
print(f"   → Programa de pontos/rewards para prescrições recorrentes")

print(f"\n💰 ROI ESPERADO:")
churn_target = churn_pct * 0.2  # 20% de retenção dos churned
print(f"   → Se converter 20% do churn em clientes recorrentes: +{churn_target:.0f} pacientes retidos")

print("\n" + "█" * 100)
print("3️⃣  ESPECIALIDADES COM BAIXO ENGAJAMENTO (Impact Médio)")
print("█" * 100)

low_engage = df_esp_kpi[df_esp_kpi['open_rate_pct'] < 50]
print(f"\n⚠️ PROBLEMA:")
print(f"   → {len(low_engage)} especialidades com open rate < 50%")
for _, row in low_engage.head(3).iterrows():
    print(f"     • {row['especialidade']}: {row['open_rate_pct']}% open rate | {row['conversao_pct']}% conversão")

print(f"\n✅ SOLUÇÃO:")
print(f"   → Contato direto com médicos destas especialidades")
print(f"   → Oferecer suporte de integração (API, treinamento)")
print(f"   → Campanhas de awareness sobre benefícios da plataforma")

print("\n" + "█" * 100)
print("4️⃣  GEOGRAFIA: EXPANSÃO E RETENÇÃO REGIONAL (Impact Médio-Alto)")
print("█" * 100)

low_or_states = df_or_estado[df_or_estado['open_rate_pct'] < 45]
print(f"\n⚠️ PROBLEMA:")
print(f"   → {len(low_or_states)} estados com open rate < 45%:")
for _, row in low_or_states.head(3).iterrows():
    print(f"     • {row['estado']}: {row['open_rate_pct']}% (potencial de +{row['prescricoes'] * (50 - row['open_rate_pct'])/100:.0f} aberturas)")

print(f"\n✅ SOLUÇÃO:")
print(f"   → Parceria com farmácias regionais para distribuição/notificação")
print(f"   → Suporte multilíngue (estado-específico)")
print(f"   → Oferta de canal local (pick-up na farmácia)")

print("\n" + "█" * 100)
print("5️⃣  CANAL: MARKETPLACE vs FARMÁCIA FÍSICA (Impact Alto)")
print("█" * 100)

mp = df_canal[df_canal['canal_final'] == 'marketplace']
ff = df_canal[df_canal['canal_final'] == 'farmacia fisica']

print(f"\n📊 PERFORMANCE ATUAL:")
if not mp.empty:
    print(f"   • Marketplace: {mp['conversao_pct'].values[0]}% conversão | {mp['vendidas'].values[0]:,} vendas")
if not ff.empty:
    print(f"   • Farmácia Física: {ff['conversao_pct'].values[0]}% conversão | {ff['vendidas'].values[0]:,} vendas")

best_channel = mp if (not mp.empty and not ff.empty and mp['conversao_pct'].values[0] > ff['conversao_pct'].values[0]) else ff
print(f"\n✅ SOLUÇÃO:")
print(f"   → Aumentar investimento em UX/marketing do canal de MAIOR conversão")
print(f"   → Remover barreiras para conversão do canal com menor performance")
print(f"   → A/B test de fluxos de checkout")

print("\n" + "█" * 100)
print("6️⃣  SEGMENTAÇÃO: OPORTUNIDADES POR FAIXA ETÁRIA (Impact Médio)")
print("█" * 100)

best_age_group = df_idade_kpi.loc[df_idade_kpi['open_rate_pct'].idxmax()]
worst_age_group = df_idade_kpi.loc[df_idade_kpi['open_rate_pct'].idxmin()]

print(f"\n📊 PERFORMANCE POR FAIXA ETÁRIA:")
print(f"   • MELHOR: {best_age_group['faixa_etaria']} ({best_age_group['open_rate_pct']}% open rate)")
print(f"   • PIOR: {worst_age_group['faixa_etaria']} ({worst_age_group['open_rate_pct']}% open rate)")

print(f"\n✅ SOLUÇÃO:")
print(f"   → Campanhas ESPECÍFICAS por faixa etária")
print(f"   → Para {worst_age_group['faixa_etaria']}: interface móvel otimizada + suporte telefônico")
print(f"   → Para {best_age_group['faixa_etaria']}: upsell (marketplace integrado, cashback)")

print("\n" + "█" * 100)
print("📋 RESUMO EXECUTIVO")
print("█" * 100)

summary = f"""
┌─ SAÚDE+ ANALYTICS SUMMARY ─────────────────────────────────────────────────────────────────┐
│                                                                                               │
│ VOLUME:           {df_funnel['emitidas'].values[0]:>10,} prescrições  |  {total_pacs_churn:>10,} pacientes  |  {pacientes_total['prescricoes_por_paciente'].values[0]:>8.1f} prescr/pac
│ OPEN RATE:        {df_funnel['open_rate_pct'].values[0]:>10.1f}%  |  ⚠️ {not_opened:,} não abertas                                
│ CONVERSÃO:        {df_funnel['conversao_pct_total'].values[0]:>10.1f}%  |  {df_funnel['conversao_pct_visualizadas'].values[0]:>8.1f}% (visualizadas→vendidas)   
│                                                                                               │
│ 🚨 CRÍTICO:       {churn_pct:,} pacientes em risco de churn (1 prescrição apenas)               
│ ⚠️  ATENÇÃO:       {len(low_engage)} especialidades com baixo engajamento                        
│ 💡 OPORTUNIDADE:  +{int(potential_conversion)} prescrições potenciais com notificação push              
│                                                                                               │
│ ✅ AÇÕES PRIORITÁRIAS (Roadmap de 30 dias):                                                 │
│    1. Implementar notificação SMS/WhatsApp (7 dias) → Impacto: +10-20% conversão            │
│    2. Lançar programa de retenção de pacientes (14 dias)                                     │
│    3. Otimizar UX do canal com melhor conversão (14 dias)                                    │
│    4. Campanha regional de awareness (21 dias)                                               │
│                                                                                               │
└───────────────────────────────────────────────────────────────────────────────────────────────┘
"""
print(summary)

🎯 Q7 - INSIGHTS OPERACIONAIS & RECOMENDAÇÕES ESTRATÉGICAS

████████████████████████████████████████████████████████████████████████████████████████████████████
1️⃣  ATIVAÇÃO IMEDIATA: NOTIFICAÇÃO PUSH (Impact Alto | Effort Baixo)
████████████████████████████████████████████████████████████████████████████████████████████████████

📱 PROBLEMA:
   → 35,135 prescrições (49.6%) não foram ABERTAS
   → Falta de notificação = perda de oportunidade de conversão

✅ SOLUÇÃO:
   → Implementar SMS/WhatsApp para receitas não abertas (6-24h após emissão)
   → Personalizado por canal (marketplace vs farmácia física)
   → A/B test com horários de envio

💰 ROI ESPERADO:
   → Se 50% das não-abertas virem a abrir: +17568 visualizações
   → Com conversão média de 12.9%, ganho: ~2266 vendas adicionais

████████████████████████████████████████████████████████████████████████████████████████████████████
2️⃣  RETENÇÃO CRÍTICA: PROGRAMA DE FIDELIZAÇÃO (Impact Crítico)
███████████████████████████████████████████

---
## Conclusão & Próximos Passos

### 📊 O que Aprendemos:

1. **Volume & Sazonalidade**: Há padrão claro semanal → otimizar recursos
2. **Pacientes**: Alto churn (uma única prescrição) → foco em retenção
3. **Especialidades**: Performance desigual → suporte diferenciado
4. **Open Rate**: Baixo (~45%) → OPORTUNIDADE de notificação
5. **Conversão**: Performance por canal → investir no winner
6. **Medicamentos**: Top 20 concentram 80% do volume
7. **Segmentação**: Faixa etária é preditor de comportamento

### 🎯 Recomendações Finais:

| Prioridade | Ação | Timeline | ROI Esperado |
|---|---|---|---|
| 🔴 CRÍTICO | Notificação Push (não-abertas) | 7 dias | +10-20% conversão |
| 🔴 CRÍTICO | Programa retenção de pacientes | 14 dias | -50% churn |
| 🟠 ALTO | Otimizar UX melhor canal | 14 dias | +5-10% conversão |
| 🟠 ALTO | Suporte para especialidades baixas | 21 dias | +2-5% open rate |
| 🟡 MÉDIO | Expansão regional | 30 dias | Novos mercados |

### 📈 KPIs a Monitorar (Dashboard):

- **Open Rate** (Target: 60%+)
- **Conversão** (Target: 10-15%)
- **Churn Rate** (Target: <30%)
- **Tempo até conversão** (Target: <24h)
- **Taxa de retenção** (Target: 70%+ recorrentes)

---
## 📝 Notas Técnicas

**Stack Utilizado:**
- **Banco de Dados**: DuckDB (zero-config, análise rápida)
- **Processamento**: Pandas DataFrames
- **Visualização**: Plotly (interativo)
- **Linguagem**: Python 3.x

**Como Executar:**
```bash
cd /path/to/saude_dashboard
pip install duckdb pandas plotly jupyter
jupyter notebook saudemais_analise.ipynb
```

**Estrutura de Dados:**
- `prescricaomedicamento.csv`: Nível de medicamento (1 linha = 1 med em 1 receita)
- `medicamentos.csv`: Catálogo de medicamentos
- `medicos.csv`: Registro de profissionais

**Adaptações para Produção:**
- Substituir DuckDB por Snowflake/BigQuery
- Criar DataWarehouse com tabelas aggregated
- Automatizar relatórios com Airflow/dbt
- Integrar com BI Tool (Looker/Tableau)

**Limitações do Dataset:**
- Período: específico no CSV (verificar datas)
- Sem dados de pricing
- Sem dados de feedback de pacientes
- Sem dados de custos operacionais

In [12]:
print("""
╔════════════════════════════════════════════════════════════════════════════════════════╗
║                   ✅ ANÁLISE SAÚDE+ CONCLUÍDA COM SUCESSO                             ║
║                                                                                         ║
║  7 Perguntas Respondidas  |  10+ Visualizações  |  5 Ações Recomendadas               ║
║                                                                                         ║
║  📊 Arquivo: saudemais_analise.ipynb                                                   ║
║  📁 Localização: /Users/bm/Desktop/saude_dashboard/                                    ║
║                                                                                         ║
║  Próximo Passo: Exportar insights para apresentação executiva                          ║
╚════════════════════════════════════════════════════════════════════════════════════════╝
""")


╔════════════════════════════════════════════════════════════════════════════════════════╗
║                   ✅ ANÁLISE SAÚDE+ CONCLUÍDA COM SUCESSO                             ║
║                                                                                         ║
║  7 Perguntas Respondidas  |  10+ Visualizações  |  5 Ações Recomendadas               ║
║                                                                                         ║
║  📊 Arquivo: saudemais_analise.ipynb                                                   ║
║  📁 Localização: /Users/bm/Desktop/saude_dashboard/                                    ║
║                                                                                         ║
║  Próximo Passo: Exportar insights para apresentação executiva                          ║
╚════════════════════════════════════════════════════════════════════════════════════════╝

